##
            TAVILY [WEB-SEARCH] 
                ||      Tool Call
                ||
                VV
    START -> CHATBOT -> END
                ^^
                ||
            LLM-Prompt

Making the websearch tool integration with TAVILY for web-search and state awarness

In [5]:
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


In [6]:
from langchain_ollama import ChatOllama


# Load local model
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0,
)

In [7]:
# Define the state
class ChatState(TypedDict):
    user_input: str
    response: str

In [9]:
# Node function
def chatbot(state: ChatState):
    answer = llm.invoke(state["user_input"])

    return {
        "response": answer.content
    }


In [10]:
# Build graph
builder = StateGraph(ChatState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()


In [11]:
# Run chatbot
while True:
    question = input("You: ")

    if question.lower() in ["exit", "quit"]:
        break

    result = graph.invoke({
        "user_input": question
    })

    print("Bot:", result["response"])

Bot: Hello! I’m doing great, thank you for asking. How can I help you today?


## Chat bot with tools

In [4]:
from langchain_tavily import TavilySearch

In [12]:
tool=TavilySearch(max_results=2)
tool.innvoke("What is the latest news about AI?")

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error